# Feature engineering — `cohort_features` v1

Phase C of the production workflow. One SQL cell per feature family, each producing a BigQuery view keyed `(subject_id, hadm_id)`. A final cell assembles `readmission.cohort_features` = `cohort_labeled` ⨝ `cohort_splits` ⨝ all seven feature views. Every later model run (`logreg-v1`, `gbt-v1`, …) reads from this single table and logs `features_table` + `features_version` (= `"v1"`) as run params.

**Default time anchor is `dischtime` of the index admission** (matches HOSPITAL / CMS HRRP framing and Phase A's prediction-time decision). Labs enforce `charttime <= dischtime` strictly. Prior-utilization features look strictly before `admittime`. Vitals are the one exception: they anchor to `icustays.outtime` (end of ICU stay), since `vitalsign` is ICU-only and most patients are stepped down before hospital discharge — anchoring to `dischtime` would yield ~3% coverage instead of ~17%.

**Missingness is not handled here.** Views emit raw values; nulls flow through. TFDV runs against `cohort_features` in Phase D and the imputation policy is decided afterwards (issue #9 in `docs/open_questions.md`).

We lean on `mimiciv_3_1_derived` wherever it removes itemid bookkeeping: Charlson comorbidities, the `vitalsign` pivot of `chartevents`, the `chemistry` / `complete_blood_count` pivots of `labevents`, and the standard ICU severity scores (`sofa`, `oasis`) and `ventilation` flag. These are the same tables built and maintained by the MIT-LCP `mimic-code` repository.

## Feature schema (~28 features across seven families)

| Family            | Source                                              | Time anchor                       | Features                                                                                                  |
|-------------------|-----------------------------------------------------|-----------------------------------|-----------------------------------------------------------------------------------------------------------|
| Demographics      | `cohort_admissions`                                 | static                            | `age_at_admission`, `gender`, `race`, `marital_status`, `insurance`, `language`, `anchor_year_group`      |
| Index admin       | `cohort_admissions` + `icustays` + `services`       | index `admittime`/`dischtime`     | `admission_type`, `admission_location`, `discharge_location`, `los_days`, `discharged_on_weekend`, `arrived_via_ed`, `icu_los_days`, `icu_stay_count`, `n_services` |
| Comorbidities     | `derived.charlson`                                  | index `dischtime`                 | `charlson_comorbidity_index` + 17 per-condition flags + `age_score`                                       |
| Prior utilization | `hosp.admissions` + `ed.edstays`                    | strictly `<` index `admittime`    | `prior_admit_count_365`, `prior_los_sum_365`, `days_since_last_discharge`, `prior_ed_count_180`           |
| Vitals (24h)      | `derived.vitalsign` (pivoted `chartevents`)         | `[icu_outtime − 24h, icu_outtime]`| mean / min / max / last for HR, SBP, DBP, RR, SpO2, temperature                                           |
| Last labs         | `derived.chemistry` + `derived.complete_blood_count`| last value `≤` index `dischtime`  | sodium, potassium, creatinine, BUN, glucose, hemoglobin, WBC, platelets                                   |
| ICU severity      | `derived.sofa` + `derived.oasis` + `derived.ventilation` | over the index stay's ICU stays | `sofa_max`, `oasis_max`, `received_ventilation`                                                           |

Each family ends with a small leakage / coverage diagnostic. Vitals and severity are non-universal — non-ICU patients legitimately have nulls — so the diagnostics also report cohort coverage.

In [1]:
import os, sys
_HERE = os.path.dirname(os.path.abspath("__file__"))
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)

import pandas as pd
from google.cloud import bigquery

from src import config

bq = bigquery.Client(project=config.PROJECT_ID, location=config.BQ_LOCATION)

# Local view-name constants (internal artifacts; only `FEATURES_TABLE`
# is consumed downstream).
V_DEMO  = f"{config.BQ_DATASET_FQN}.v_features_demographics"
V_ADMIN = f"{config.BQ_DATASET_FQN}.v_features_index_admin"
V_COMO  = f"{config.BQ_DATASET_FQN}.v_features_comorbidities"
V_PRIOR = f"{config.BQ_DATASET_FQN}.v_features_prior_utilization"
V_VIT   = f"{config.BQ_DATASET_FQN}.v_features_vitals_24h"
V_LAB   = f"{config.BQ_DATASET_FQN}.v_features_labs_last"
V_SEV   = f"{config.BQ_DATASET_FQN}.v_features_severity"

def run(sql: str) -> bigquery.QueryJob:
    """Execute a DDL/DML statement and block until done."""
    job = bq.query(sql)
    job.result()
    return job

def show(sql: str) -> pd.DataFrame:
    """Run a SELECT and return a DataFrame (also pretty-prints it)."""
    df = bq.query(sql).result().to_dataframe()
    print(df.to_string(index=False))
    return df

## Family 1 — Demographics

Static patient and admission-context attributes. All columns are already promoted onto `cohort_admissions` by Phase A, so this view is a thin projection. Diagnostic confirms 1:1 with the cohort.

In [2]:
demographics_sql = f"""
CREATE OR REPLACE VIEW `{V_DEMO}` AS
SELECT
  subject_id,
  hadm_id,
  age_at_admission,
  gender,
  race,
  marital_status,
  insurance,
  language,
  anchor_year_group
FROM `{config.COHORT_TABLE}`
"""
run(demographics_sql)

show(f"""
SELECT
  (SELECT COUNT(*) FROM `{V_DEMO}`)               AS n_view_rows,
  (SELECT COUNT(*) FROM `{config.COHORT_TABLE}`)  AS n_cohort_rows
""")

 n_view_rows  n_cohort_rows
      406958         406958


,n_view_rows,n_cohort_rows
0,406958,406958


## Family 2 — Index-stay administrative

Stay-level operational signal known at discharge: admission type, locations, length of stay, weekend-discharge flag, ED-arrival flag, ICU exposure during the stay, and number of distinct services that saw the patient. ICU and service counts come from `mimiciv_3_1_icu.icustays` and `mimiciv_3_1_hosp.services` aggregated to `hadm_id`.

In [3]:
index_admin_sql = f"""
CREATE OR REPLACE VIEW `{V_ADMIN}` AS
WITH icu AS (
  SELECT hadm_id,
         SUM(los)     AS icu_los_days,
         COUNT(*)     AS icu_stay_count
  FROM `{config.MIMIC_ICU}.icustays`
  GROUP BY hadm_id
),
svc AS (
  SELECT hadm_id,
         COUNT(DISTINCT curr_service) AS n_services
  FROM `{config.MIMIC_HOSP}.services`
  GROUP BY hadm_id
)
SELECT
  c.subject_id,
  c.hadm_id,
  c.admission_type,
  c.admission_location,
  c.discharge_location,
  c.los_hours / 24.0                                  AS los_days,
  EXTRACT(DAYOFWEEK FROM c.dischtime) IN (1, 7)       AS discharged_on_weekend,
  c.edregtime IS NOT NULL                             AS arrived_via_ed,
  COALESCE(icu.icu_los_days, 0)                       AS icu_los_days,
  COALESCE(icu.icu_stay_count, 0)                     AS icu_stay_count,
  COALESCE(svc.n_services, 1)                         AS n_services
FROM `{config.COHORT_TABLE}` c
LEFT JOIN icu USING (hadm_id)
LEFT JOIN svc USING (hadm_id)
"""
run(index_admin_sql)

show(f"""
SELECT
  COUNT(*)                                              AS n_view_rows,
  COUNTIF(icu_stay_count > 0)                           AS n_with_icu,
  ROUND(AVG(los_days), 2)                               AS mean_los_days,
  ROUND(AVG(IF(discharged_on_weekend, 1, 0)), 3)        AS pct_weekend_discharge,
  ROUND(AVG(IF(arrived_via_ed,        1, 0)), 3)        AS pct_arrived_via_ed
FROM `{V_ADMIN}`
""")

 n_view_rows  n_with_icu  mean_los_days  pct_weekend_discharge  pct_arrived_via_ed
      406958       70655           5.67                  0.286               0.647


,n_view_rows,n_with_icu,mean_los_days,pct_weekend_discharge,pct_arrived_via_ed
0,406958,70655,5.67,0.286,0.647


## Family 3 — Comorbidities (Charlson)

Sourced from `mimiciv_3_1_derived.charlson`, which already maps the index admission's `diagnoses_icd` to the 17 Charlson conditions and emits an `age_score` plus the composite `charlson_comorbidity_index`. Joined `LEFT` so cohort rows with no Charlson row keep nulls (rare but possible for very short or atypical encounters); coverage is reported by the diagnostic.

In [4]:
comorbidities_sql = f"""
CREATE OR REPLACE VIEW `{V_COMO}` AS
SELECT
  c.subject_id,
  c.hadm_id,
  ch.age_score,
  ch.myocardial_infarct,
  ch.congestive_heart_failure,
  ch.peripheral_vascular_disease,
  ch.cerebrovascular_disease,
  ch.dementia,
  ch.chronic_pulmonary_disease,
  ch.rheumatic_disease,
  ch.peptic_ulcer_disease,
  ch.mild_liver_disease,
  ch.diabetes_without_cc,
  ch.diabetes_with_cc,
  ch.paraplegia,
  ch.renal_disease,
  ch.malignant_cancer,
  ch.severe_liver_disease,
  ch.metastatic_solid_tumor,
  ch.aids,
  ch.charlson_comorbidity_index
FROM `{config.COHORT_TABLE}` c
LEFT JOIN `{config.MIMIC_DERIVED}.charlson` ch
  USING (subject_id, hadm_id)
"""
run(comorbidities_sql)

show(f"""
SELECT
  COUNT(*)                                              AS n_view_rows,
  COUNTIF(charlson_comorbidity_index IS NOT NULL)       AS n_with_charlson,
  ROUND(AVG(charlson_comorbidity_index), 2)             AS mean_cci,
  ROUND(AVG(IF(congestive_heart_failure = 1, 1, 0)), 3) AS pct_chf,
  ROUND(AVG(IF(diabetes_with_cc        = 1, 1, 0)), 3)  AS pct_dm_with_cc
FROM `{V_COMO}`
""")

 n_view_rows  n_with_charlson  mean_cci  pct_chf  pct_dm_with_cc
      406958           406958      3.89    0.175           0.102


,n_view_rows,n_with_charlson,mean_cci,pct_chf,pct_dm_with_cc
0,406958,406958,3.89,0.175,0.102


## Family 4 — Prior utilization

Counts of prior inpatient admissions and ED visits relative to the index admission. Sourced from the **full** `admissions` table (not just the cohort) and `mimiciv_ed.edstays`, since a patient's prior non-eligible visits still carry signal. All windows enforce strict `<` `admittime` of the index stay — this is the project's hard rule on prior-utilization leakage.

* `prior_admit_count_365` — distinct prior `hadm_id` with `dischtime` in `[admittime − 365d, admittime)`.
* `prior_los_sum_365` — sum of LOS in days for those admissions.
* `days_since_last_discharge` — days from the most recent prior discharge to the index `admittime` (NULL for first-ever admit).
* `prior_ed_count_180` — distinct prior ED stays with `outtime` in `[admittime − 180d, admittime)`.

In [6]:
prior_utilization_sql = f"""
CREATE OR REPLACE VIEW `{V_PRIOR}` AS
WITH prior_admit AS (
  SELECT
    c.subject_id,
    c.hadm_id,
    COUNTIF(TIMESTAMP_DIFF(c.admittime, a.dischtime, DAY) BETWEEN 0 AND 365)
      AS prior_admit_count_365,
    SUM(CASE
          WHEN TIMESTAMP_DIFF(c.admittime, a.dischtime, DAY) BETWEEN 0 AND 365
          THEN TIMESTAMP_DIFF(a.dischtime, a.admittime, HOUR) / 24.0
          ELSE 0
        END) AS prior_los_sum_365,
    MIN(TIMESTAMP_DIFF(c.admittime, a.dischtime, DAY))
      AS days_since_last_discharge
  FROM `{config.COHORT_TABLE}` c
  LEFT JOIN `{config.MIMIC_HOSP}.admissions` a
    ON a.subject_id = c.subject_id
   AND a.hadm_id   != c.hadm_id
   AND a.dischtime  < c.admittime
  GROUP BY c.subject_id, c.hadm_id
),
prior_ed AS (
  SELECT
    c.subject_id,
    c.hadm_id,
    COUNTIF(TIMESTAMP_DIFF(c.admittime, e.outtime, DAY) BETWEEN 0 AND 180)
      AS prior_ed_count_180
  FROM `{config.COHORT_TABLE}` c
  LEFT JOIN `{config.MIMIC_ED}.edstays` e
    ON e.subject_id = c.subject_id
   AND e.outtime    < c.admittime
  GROUP BY c.subject_id, c.hadm_id
)
SELECT
  c.subject_id,
  c.hadm_id,
  COALESCE(pa.prior_admit_count_365, 0)  AS prior_admit_count_365,
  COALESCE(pa.prior_los_sum_365,     0)  AS prior_los_sum_365,
  pa.days_since_last_discharge           AS days_since_last_discharge,
  COALESCE(pe.prior_ed_count_180,    0)  AS prior_ed_count_180
FROM `{config.COHORT_TABLE}` c
LEFT JOIN prior_admit pa USING (subject_id, hadm_id)
LEFT JOIN prior_ed    pe USING (subject_id, hadm_id)
"""
run(prior_utilization_sql)

# Distribution summary.
show(f"""
SELECT
  COUNT(*)                                  AS n_view_rows,
  COUNTIF(prior_admit_count_365 > 0)        AS n_with_prior_admit,
  COUNTIF(prior_ed_count_180    > 0)        AS n_with_prior_ed,
  ROUND(AVG(prior_admit_count_365), 2)      AS mean_prior_admit_count_365,
  ROUND(AVG(prior_ed_count_180), 2)         AS mean_prior_ed_count_180,
  ROUND(AVG(days_since_last_discharge), 1)  AS mean_days_since_last_dc
FROM `{V_PRIOR}`
""")

# Leakage check: mirrors the feature SQL's strict `<` filter exactly.
# Any source row that contributed to the aggregation must satisfy
# dischtime/outtime < index admittime. Both counts must be 0.
print()
print("Leakage check (must be 0 / 0):")
show(f"""
WITH leak_admit AS (
  SELECT COUNTIF(a.dischtime >= c.admittime) AS n
  FROM `{config.COHORT_TABLE}` c
  JOIN `{config.MIMIC_HOSP}.admissions` a
    ON a.subject_id = c.subject_id
   AND a.hadm_id   != c.hadm_id
   AND a.dischtime  < c.admittime
  WHERE TIMESTAMP_DIFF(c.admittime, a.dischtime, DAY) BETWEEN 0 AND 365
),
leak_ed AS (
  SELECT COUNTIF(e.outtime >= c.admittime) AS n
  FROM `{config.COHORT_TABLE}` c
  JOIN `{config.MIMIC_ED}.edstays` e
    ON e.subject_id = c.subject_id
   AND e.outtime    < c.admittime
  WHERE TIMESTAMP_DIFF(c.admittime, e.outtime, DAY) BETWEEN 0 AND 180
)
SELECT (SELECT n FROM leak_admit) AS leak_admit_rows,
       (SELECT n FROM leak_ed)    AS leak_ed_rows
""")

 n_view_rows  n_with_prior_admit  n_with_prior_ed  mean_prior_admit_count_365  mean_prior_ed_count_180  mean_days_since_last_dc
      406958              180775            79324                        1.19                      0.4                    367.1

Leakage check (must be 0 / 0):
 leak_admit_rows  leak_ed_rows
               0             0


,leak_admit_rows,leak_ed_rows
0,0,0


## Family 5 — Vitals (last 24 h of ICU care)

Built on `mimiciv_3_1_derived.vitalsign`, the `mimic-code` pivot of `chartevents` into named columns (`heart_rate`, `sbp`, `sbp_ni`, `dbp`, `dbp_ni`, `resp_rate`, `spo2`, `temperature`).

**Time anchor:** `icustays.outtime` (end of each ICU stay), not the hospital `dischtime`. Rationale: `vitalsign` only contains ICU-monitored data; most ICU patients are stepped down to a floor before hospital discharge, so a `[dischtime − 24h, dischtime]` window misses them. Anchoring to `outtime` captures the "discharge-from-ICU physiology snapshot" that readmission-prediction work typically uses, and brings coverage up from ~3% (24 h before hospital discharge) to the cohort's ICU rate (~17%). For admissions with multiple ICU stays, events from any stay's last 24 h are aggregated together; the `*_last_24h` columns then resolve to the most recent reading overall — i.e. the last reading before final ICU discharge for that admission.

We coalesce arterial and non-invasive BP (`sbp`, `sbp_ni`) so a single `sbp_*` column is emitted regardless of monitoring modality. For each of HR, SBP, DBP, RR, SpO2, and temperature we emit `mean / min / max / last_24h`.

This family remains **non-universal** — only patients with an ICU stay appear. The diagnostic reports cohort coverage; TFDV will quantify per-feature missingness in Phase D.

In [11]:
vitals_sql = f"""
CREATE OR REPLACE VIEW `{V_VIT}` AS
WITH events AS (
  SELECT
    c.subject_id,
    c.hadm_id,
    vs.charttime,
    vs.heart_rate,
    COALESCE(vs.sbp, vs.sbp_ni)        AS sbp,
    COALESCE(vs.dbp, vs.dbp_ni)        AS dbp,
    vs.resp_rate,
    vs.spo2,
    vs.temperature
  FROM `{config.COHORT_TABLE}` c
  JOIN `{config.MIMIC_ICU}.icustays`     i  USING (subject_id, hadm_id)
  JOIN `{config.MIMIC_DERIVED}.vitalsign` vs USING (stay_id)
  WHERE vs.charttime BETWEEN TIMESTAMP_SUB(i.outtime, INTERVAL 24 HOUR)
                         AND i.outtime
    AND i.outtime <= c.dischtime
),
agg AS (
  SELECT
    subject_id, hadm_id,
    AVG(heart_rate) AS hr_mean_24h,
    MIN(heart_rate) AS hr_min_24h,
    MAX(heart_rate) AS hr_max_24h,
    ARRAY_AGG(heart_rate IGNORE NULLS ORDER BY charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS hr_last_24h,
    AVG(sbp) AS sbp_mean_24h,
    MIN(sbp) AS sbp_min_24h,
    MAX(sbp) AS sbp_max_24h,
    ARRAY_AGG(sbp IGNORE NULLS ORDER BY charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS sbp_last_24h,
    AVG(dbp) AS dbp_mean_24h,
    MIN(dbp) AS dbp_min_24h,
    MAX(dbp) AS dbp_max_24h,
    ARRAY_AGG(dbp IGNORE NULLS ORDER BY charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS dbp_last_24h,
    AVG(resp_rate) AS rr_mean_24h,
    MIN(resp_rate) AS rr_min_24h,
    MAX(resp_rate) AS rr_max_24h,
    ARRAY_AGG(resp_rate IGNORE NULLS ORDER BY charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS rr_last_24h,
    AVG(spo2) AS spo2_mean_24h,
    MIN(spo2) AS spo2_min_24h,
    MAX(spo2) AS spo2_max_24h,
    ARRAY_AGG(spo2 IGNORE NULLS ORDER BY charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS spo2_last_24h,
    CAST(AVG(temperature) AS FLOAT64) AS temp_mean_24h,
    CAST(MIN(temperature) AS FLOAT64) AS temp_min_24h,
    CAST(MAX(temperature) AS FLOAT64) AS temp_max_24h,
    CAST(ARRAY_AGG(temperature IGNORE NULLS ORDER BY charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS FLOAT64) AS temp_last_24h
  FROM events
  GROUP BY subject_id, hadm_id
)
SELECT
  c.subject_id,
  c.hadm_id,
  agg.* EXCEPT(subject_id, hadm_id)
FROM `{config.COHORT_TABLE}` c
LEFT JOIN agg USING (subject_id, hadm_id)
"""
run(vitals_sql)

# Coverage diagnostic. Vitals are anchored to ICU outtime, so expected
# coverage ≈ cohort ICU rate (~17%). Should track Family 7's coverage.
show(f"""
SELECT
  COUNT(*)                              AS n_view_rows,
  COUNTIF(hr_mean_24h   IS NOT NULL)    AS n_with_hr,
  COUNTIF(sbp_mean_24h  IS NOT NULL)    AS n_with_sbp,
  COUNTIF(spo2_mean_24h IS NOT NULL)    AS n_with_spo2,
  ROUND(SAFE_DIVIDE(COUNTIF(hr_mean_24h IS NOT NULL), COUNT(*)), 3) AS coverage_hr
FROM `{V_VIT}`
""")

 n_view_rows  n_with_hr  n_with_sbp  n_with_spo2  coverage_hr
      406958      63925       63881        63871        0.157


,n_view_rows,n_with_hr,n_with_sbp,n_with_spo2,coverage_hr
0,406958,63925,63881,63871,0.157


## Family 6 — Last pre-discharge labs

Built on `mimiciv_3_1_derived.chemistry` (sodium, potassium, creatinine, BUN, glucose) and `mimiciv_3_1_derived.complete_blood_count` (hemoglobin, WBC, platelets) — both are `mimic-code` pivots of `labevents` keyed on `(hadm_id, charttime)`. For each lab we take the most recent non-null value where `charttime <= dischtime`. Coverage is reported per lab; nulls flow through unchanged.

In [8]:
labs_sql = f"""
CREATE OR REPLACE VIEW `{V_LAB}` AS
WITH chem AS (
  SELECT
    c.subject_id, c.hadm_id, ch.charttime,
    ch.sodium, ch.potassium, ch.creatinine, ch.bun, ch.glucose
  FROM `{config.COHORT_TABLE}` c
  JOIN `{config.MIMIC_DERIVED}.chemistry` ch
    ON ch.subject_id = c.subject_id
   AND ch.hadm_id    = c.hadm_id
   AND ch.charttime <= c.dischtime
),
last_chem AS (
  SELECT
    subject_id, hadm_id,
    ARRAY_AGG(sodium     IGNORE NULLS ORDER BY charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS sodium_last,
    ARRAY_AGG(potassium  IGNORE NULLS ORDER BY charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS potassium_last,
    ARRAY_AGG(creatinine IGNORE NULLS ORDER BY charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS creatinine_last,
    ARRAY_AGG(bun        IGNORE NULLS ORDER BY charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS bun_last,
    ARRAY_AGG(glucose    IGNORE NULLS ORDER BY charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS glucose_last
  FROM chem
  GROUP BY subject_id, hadm_id
),
cbc AS (
  SELECT
    c.subject_id, c.hadm_id, b.charttime,
    b.hemoglobin, b.wbc, b.platelet
  FROM `{config.COHORT_TABLE}` c
  JOIN `{config.MIMIC_DERIVED}.complete_blood_count` b
    ON b.subject_id = c.subject_id
   AND b.hadm_id    = c.hadm_id
   AND b.charttime <= c.dischtime
),
last_cbc AS (
  SELECT
    subject_id, hadm_id,
    ARRAY_AGG(hemoglobin IGNORE NULLS ORDER BY charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS hemoglobin_last,
    ARRAY_AGG(wbc        IGNORE NULLS ORDER BY charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS wbc_last,
    ARRAY_AGG(platelet   IGNORE NULLS ORDER BY charttime DESC LIMIT 1)[SAFE_OFFSET(0)] AS platelets_last
  FROM cbc
  GROUP BY subject_id, hadm_id
)
SELECT
  c.subject_id,
  c.hadm_id,
  lc.sodium_last,
  lc.potassium_last,
  lc.creatinine_last,
  lc.bun_last,
  lc.glucose_last,
  lb.hemoglobin_last,
  lb.wbc_last,
  lb.platelets_last
FROM `{config.COHORT_TABLE}` c
LEFT JOIN last_chem lc USING (subject_id, hadm_id)
LEFT JOIN last_cbc  lb USING (subject_id, hadm_id)
"""
run(labs_sql)

show(f"""
SELECT
  COUNT(*)                                  AS n_view_rows,
  COUNTIF(sodium_last     IS NOT NULL)      AS n_with_sodium,
  COUNTIF(creatinine_last IS NOT NULL)      AS n_with_creatinine,
  COUNTIF(hemoglobin_last IS NOT NULL)      AS n_with_hemoglobin,
  COUNTIF(glucose_last    IS NOT NULL)      AS n_with_glucose,
  ROUND(SAFE_DIVIDE(COUNTIF(sodium_last IS NOT NULL), COUNT(*)), 3) AS coverage_sodium
FROM `{V_LAB}`
""")

 n_view_rows  n_with_sodium  n_with_creatinine  n_with_hemoglobin  n_with_glucose  coverage_sodium
      406958         347575             352455             360972          345937            0.854


,n_view_rows,n_with_sodium,n_with_creatinine,n_with_hemoglobin,n_with_glucose,coverage_sodium
0,406958,347575,352455,360972,345937,0.854


## Family 7 — ICU severity (bonus)

ICU-only severity signal aggregated across all ICU stays of the index admission. Three features:

* `sofa_max` — peak `sofa_24hours` from `mimiciv_3_1_derived.sofa` (rolling 24-h SOFA at hourly resolution).
* `oasis_max` — max OASIS score from `mimiciv_3_1_derived.oasis` across the admission's ICU stays.
* `received_ventilation` — boolean: any row in `mimiciv_3_1_derived.ventilation` for any ICU stay of the admission.

Non-ICU patients have nulls for `sofa_max` and `oasis_max`; `received_ventilation` is coalesced to `FALSE` (absence of evidence is taken as no ventilation, matching how the table is built).

In [9]:
severity_sql = f"""
CREATE OR REPLACE VIEW `{V_SEV}` AS
WITH stays AS (
  SELECT subject_id, hadm_id, stay_id
  FROM `{config.MIMIC_ICU}.icustays`
),
sofa_per_hadm AS (
  SELECT s.subject_id, s.hadm_id, MAX(so.sofa_24hours) AS sofa_max
  FROM stays s
  JOIN `{config.MIMIC_DERIVED}.sofa` so USING (stay_id)
  GROUP BY s.subject_id, s.hadm_id
),
oasis_per_hadm AS (
  SELECT s.subject_id, s.hadm_id, MAX(o.oasis) AS oasis_max
  FROM stays s
  JOIN `{config.MIMIC_DERIVED}.oasis` o USING (stay_id)
  GROUP BY s.subject_id, s.hadm_id
),
vent_per_hadm AS (
  SELECT s.subject_id, s.hadm_id, TRUE AS received_ventilation
  FROM stays s
  JOIN `{config.MIMIC_DERIVED}.ventilation` v USING (stay_id)
  GROUP BY s.subject_id, s.hadm_id
)
SELECT
  c.subject_id,
  c.hadm_id,
  sp.sofa_max,
  op.oasis_max,
  COALESCE(vp.received_ventilation, FALSE) AS received_ventilation
FROM `{config.COHORT_TABLE}` c
LEFT JOIN sofa_per_hadm  sp USING (subject_id, hadm_id)
LEFT JOIN oasis_per_hadm op USING (subject_id, hadm_id)
LEFT JOIN vent_per_hadm  vp USING (subject_id, hadm_id)
"""
run(severity_sql)

show(f"""
SELECT
  COUNT(*)                                                   AS n_view_rows,
  COUNTIF(sofa_max  IS NOT NULL)                             AS n_with_sofa,
  COUNTIF(oasis_max IS NOT NULL)                             AS n_with_oasis,
  COUNTIF(received_ventilation)                              AS n_received_vent,
  ROUND(AVG(sofa_max),  2)                                   AS mean_sofa_max,
  ROUND(AVG(oasis_max), 2)                                   AS mean_oasis_max,
  ROUND(SAFE_DIVIDE(COUNTIF(sofa_max IS NOT NULL), COUNT(*)), 3) AS coverage_sofa
FROM `{V_SEV}`
""")

 n_view_rows  n_with_sofa  n_with_oasis  n_received_vent  mean_sofa_max  mean_oasis_max  coverage_sofa
      406958        70653         70655            52733           4.34           29.82          0.174


,n_view_rows,n_with_sofa,n_with_oasis,n_received_vent,mean_sofa_max,mean_oasis_max,coverage_sofa
0,406958,70653,70655,52733,4.34,29.82,0.174


## Assemble `cohort_features`

Materialize `readmission.cohort_features` as the join of `cohort_labeled ⨝ cohort_splits ⨝` all seven feature views, keyed on `(subject_id, hadm_id)` and clustered by `subject_id`. One row per eligible `hadm_id`, with label, split, and the full feature vector. This is the single table downstream model code reads.

* Demographics, index admin, and prior utilization use **inner** joins (every cohort row has these by construction).
* Comorbidities, vitals, labs, and severity use **left** joins (some cohort rows legitimately have no source data).

The sanity summary at the end reports row count, column count, per-family coverage, and per-split prevalence to confirm the splits contract carries through unchanged.

In [12]:
assemble_sql = f"""
CREATE OR REPLACE TABLE `{config.FEATURES_TABLE}`
CLUSTER BY subject_id AS
SELECT
  cl.subject_id,
  cl.hadm_id,
  cl.label,
  s.split,
  d.*  EXCEPT(subject_id, hadm_id),
  ia.* EXCEPT(subject_id, hadm_id),
  co.* EXCEPT(subject_id, hadm_id),
  pu.* EXCEPT(subject_id, hadm_id),
  vt.* EXCEPT(subject_id, hadm_id),
  lb.* EXCEPT(subject_id, hadm_id),
  sv.* EXCEPT(subject_id, hadm_id)
FROM `{config.LABELED_TABLE}` cl
JOIN      `{config.SPLITS_TABLE}` s  USING (subject_id)
JOIN      `{V_DEMO}`              d  USING (subject_id, hadm_id)
JOIN      `{V_ADMIN}`             ia USING (subject_id, hadm_id)
LEFT JOIN `{V_COMO}`              co USING (subject_id, hadm_id)
JOIN      `{V_PRIOR}`             pu USING (subject_id, hadm_id)
LEFT JOIN `{V_VIT}`               vt USING (subject_id, hadm_id)
LEFT JOIN `{V_LAB}`               lb USING (subject_id, hadm_id)
LEFT JOIN `{V_SEV}`               sv USING (subject_id, hadm_id)
"""
run(assemble_sql)

table = bq.get_table(config.FEATURES_TABLE)
n_features = len(table.schema) - 4  # subject_id, hadm_id, label, split

print(f"Built {config.FEATURES_TABLE}")
print(f"  features_version: {config.FEATURES_VERSION}")
print(f"  rows:             {table.num_rows:,}")
print(f"  bytes:            {table.num_bytes / 1e6:,.1f} MB")
print(f"  n_columns:        {len(table.schema)}  ({n_features} feature columns)")
print(f"  cluster col:      subject_id")
print()

print("Row-count parity (n_features_rows must equal n_labeled_rows):")
show(f"""
SELECT
  (SELECT COUNT(*) FROM `{config.FEATURES_TABLE}`) AS n_features_rows,
  (SELECT COUNT(*) FROM `{config.LABELED_TABLE}`)  AS n_labeled_rows
""")
print()

print("Per-split row counts and prevalence (must match Phase A's contract):")
show(f"""
SELECT split,
       COUNT(*)             AS n_rows,
       ROUND(AVG(label), 4) AS prevalence
FROM `{config.FEATURES_TABLE}`
GROUP BY split
ORDER BY CASE split WHEN 'train' THEN 1 WHEN 'val' THEN 2 WHEN 'test' THEN 3 ELSE 4 END
""")
print()

print("Per-family coverage (representative column from each):")
show(f"""
SELECT
  ROUND(SAFE_DIVIDE(COUNTIF(age_at_admission           IS NOT NULL), COUNT(*)), 3) AS demographics,
  ROUND(SAFE_DIVIDE(COUNTIF(los_days                   IS NOT NULL), COUNT(*)), 3) AS index_admin,
  ROUND(SAFE_DIVIDE(COUNTIF(charlson_comorbidity_index IS NOT NULL), COUNT(*)), 3) AS comorbidities,
  ROUND(SAFE_DIVIDE(COUNTIF(prior_admit_count_365      IS NOT NULL), COUNT(*)), 3) AS prior_utilization,
  ROUND(SAFE_DIVIDE(COUNTIF(hr_mean_24h                IS NOT NULL), COUNT(*)), 3) AS vitals_24h,
  ROUND(SAFE_DIVIDE(COUNTIF(creatinine_last            IS NOT NULL), COUNT(*)), 3) AS labs_last,
  ROUND(SAFE_DIVIDE(COUNTIF(sofa_max                   IS NOT NULL), COUNT(*)), 3) AS severity
FROM `{config.FEATURES_TABLE}`
""")

Built enterprise-clinical-copilot.readmission.cohort_features
  features_version: v1
  rows:             406,958
  bytes:            182.3 MB
  n_columns:        78  (74 feature columns)
  cluster col:      subject_id

Row-count parity (n_features_rows must equal n_labeled_rows):
 n_features_rows  n_labeled_rows
          406958          406958

Per-split row counts and prevalence (must match Phase A's contract):
split  n_rows  prevalence
train  284412      0.1856
  val   60874      0.1850
 test   59792      0.1913
 demo    1880      0.1707

Per-family coverage (representative column from each):
 demographics  index_admin  comorbidities  prior_utilization  vitals_24h  labs_last  severity
          1.0          1.0            1.0                1.0       0.157      0.866     0.174


,demographics,index_admin,comorbidities,prior_utilization,vitals_24h,labs_last,severity
0,1.0,1.0,1.0,1.0,0.157,0.866,0.174
